In [ ]:
"""
Objective: Connect to the Federal Reserve API (FRED), extract monthly macroeconomic series 
(TED Spread, Unemployment, and Fed Funds), and export the consolidated dataset for the 
dynamic estimations via Rolling-Window VAR focusing on the 2008 crisis.
"""

import pandas as pd
from fredapi import Fred
import warnings
import os
os.environ['FRED_API_KEY'] = 'FRED_API_KEY'
import sys

warnings.filterwarnings('ignore')

# ==========================================
# 1. AUTHENTICATION AND FRED CONNECTION
# ==========================================
# The API key must be configured in the environment (e.g., export FRED_API_KEY='your_key')
# or placed in a .env file to prevent leaks in public repositories.
API_KEY = os.getenv('FRED_API_KEY')

if not API_KEY:
    print("[ERROR] FRED API key missing.")
    print("-> Please configure the 'FRED_API_KEY' environment variable before running.")
    sys.exit(1)

def extract_rolling_var_panel(api_key: str, output_file: str):
    """Extracts, aligns monthly frequencies, and exports the macroeconomic panel."""
    print("[*] Authenticating with the Federal Reserve (FRED) server...")
    
    try:
        fred = Fred(api_key=api_key)
        
        # Truncated window to capture the calm period, Subprime, and Post-Lehman eras
        start_date, end_date = '2000-01-01', '2015-12-31'
        
        print(f"[*] Extracting series ({start_date} to {end_date}) at monthly frequency...")
        
        # TED Spread: Fetch daily data and convert to monthly average
        ted_spread = fred.get_series('TEDRATE', observation_start=start_date, observation_end=end_date)
        ted_spread = ted_spread.resample('MS').mean()
        
        # Unemployment and Fed Funds (Monthly base)
        unrate = fred.get_series('UNRATE', observation_start=start_date, observation_end=end_date)
        fedfunds = fred.get_series('FEDFUNDS', observation_start=start_date, observation_end=end_date)
        
        # Macro Vector Consolidation
        df_var = pd.DataFrame({
            'TED_Spread': ted_spread,
            'Unemployment': unrate,
            'Fed_Funds': fedfunds
        })
        
        # Cleaning and calendar alignment
        df_var.index.name = 'Date'
        df_var.dropna(inplace=True)
        
        # Exporting
        df_var.to_csv(output_file)
        
        print("\n[SUCCESS] Rolling-Window VAR lab dataset structured and exported.")
        print(f"[*] File generated: {output_file}")
        print(f"[*] Dimensions: {df_var.shape[0]} months captured.")
        
    except ValueError as ve:
        print(f"\n[AUTHENTICATION ERROR] {ve}. Check your API key validity.")
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Communication failure with FRED: {e}")

if __name__ == "__main__":
    extract_rolling_var_panel(API_KEY, '07_clean_rolling_var_fred.csv')